## Daten laden


In [ ]:
%reload_ext autoreload

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score, recall_score, confusion_matrix
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from utils import bewerten, trainieren_und_bewerten

df = pd.read_parquet('../data/processed/credit_transformed.parquet')
df.dtypes

---
## Merkmale trennen und Kategorien in Zahlen umwandeln

In [ ]:
X = df.drop(columns='loan_status')
y = df['loan_status'].astype(int)

X['cb_person_default_on_file'] = X['cb_person_default_on_file'].astype(float)
X['kredit_einkommen'] = X['loan_amnt'] / X['person_income']
X = pd.get_dummies(X, columns=['person_home_ownership', 'loan_intent'], drop_first=True, dtype=float)

X.head()

In [ ]:
y.head()

---
## Train-Test Split 80/20

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(X_train.shape, X_test.shape)

---
## Trainings Mediane abgrenzen
Sonst würde ein Data-leakage ein zu gutes Ergebnis aka overfitting erzeugen
Eigentlich gehört dieser Schritt in 02_Datapreparation aber da es einen Mergeconflict mit Carl geben könnte, ist es besser hier aufgehoben.

Der Gedanke hinter dem Median: Die Daten zu löschen würde uns 10% des Datensatzes kosten. Wenn wir den Median in die N.A werte einfüllen, dann gibt es keine Ausreißer und wir können die Daten trotzdem noch verwenden.

Man könnte auch noch ein Schätzungsmodell für die Daten bauen aber das ist für die paar daten zu aufwendig..

In [ ]:
mediane = X_train.median()
X_train = X_train.fillna(mediane)
X_test = X_test.fillna(mediane)

X_train.isna().sum().sum(), X_test.isna().sum().sum()   # sollte (0, 0) ergeben

---
## Daten speichern

In [ ]:
from pathlib import Path

ordner = Path('../data/processed')
ordner.mkdir(parents=True, exist_ok=True)

X_train.to_parquet(ordner / 'X_train.parquet')
X_test.to_parquet(ordner / 'X_test.parquet')
y_train.to_frame().to_parquet(ordner / 'y_train.parquet')
y_test.to_frame().to_parquet(ordner / 'y_test.parquet')

---
## Modellvergleiche 


### 1. Lineare Regression

In [ ]:
model_lr = trainieren_und_bewerten('Logistische Regression', make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), LogisticRegression(max_iter=1000)), X_train, y_train, X_test, y_test)    

---
### 2. Logistic Regression

In [ ]:
model_log = trainieren_und_bewerten('Logistische Regression', make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), LogisticRegression(max_iter=1000)), X_train, y_train, X_test, y_test)

---
### 3. Naive Bayes

In [ ]:
model_nb = trainieren_und_bewerten('Naive Bayes', GaussianNB(), X_train, y_train, X_test, y_test)

---
### 4. Support Vector Machine

In [ ]:
model_svm = trainieren_und_bewerten('Support Vector Machine', SVC(probability=True), X_train, y_train, X_test, y_test)

---
### 5. Decision Tree

In [ ]:
model_tree = trainieren_und_bewerten('Entscheidungsbaum', DecisionTreeClassifier(max_depth=6, random_state=42), X_train, y_train, X_test, y_test)

---
### 6. Random Forest

In [ ]:
model_rf =trainieren_und_bewerten('Random Forest', RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42), X_train, y_train, X_test, y_test)

---
### 7. Gradient Boosting

In [ ]:
model_gb = trainieren_und_bewerten('Gradient Boosting', HistGradientBoostingClassifier(random_state=42),
                                   X_train, y_train, X_test, y_test)

---
### 8. K-Nearest-Neighbor


In [ ]:
model_knn = trainieren_und_bewerten('kNN', make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), KNeighborsClassifier(n_neighbors=15)),
                                   X_train, y_train, X_test, y_test)    

---
### 9. Multi-Layer Perceptron

In [ ]:
model_mlp = trainieren_und_bewerten('MLP', MLPClassifier(), X_train, y_train, X_test, y_test)